# Visualizes a crop and a mask together with the whole brain

This notebook makes some plotting after having generated some crops based on bounding box.µ

# Imports

In [ ]:
import os
import json
import tempfile
import glob
import shutil
import inspect
import dico_toolbox as dtx
import colorado as cld
import pcpm

from six.moves.urllib.request import urlopen
import zipfile
from soma import aims
from soma import aimsalgo
import numpy as np

In [ ]:
import cortical_tiles
print(inspect.getfile(cortical_tiles))
from cortical_tiles.brainvisa import compute_bounding_box

In [ ]:
import anatomist.notebook as ana
a = ana.Anatomist()
print(a.headless_info.__dict__)

# User-specific variables

In [ ]:
sulcus_ant = 'S.T.s.ter.asc.ant.'
sulcus_post = 'S.T.s.ter.asc.post.'

In [ ]:
side = 'L'

We now assign path names and other user-specific variables.

The source directory is where the database lies. It contains the morphologist analysis subfolder ANALYSIS/3T_morphologist


In [ ]:
src_dir = os.path.join(os.getcwd(), '../data/source/supervised')
src_dir_supervised = src_dir
src_dir = os.path.abspath(src_dir)
print(("src_dir = " + src_dir))

The target directory tgt_dir is where the files will be saved

In [ ]:
bbox_dir = os.path.join(os.getcwd(), '../data/target/bbox')
bbox_dir = os.path.abspath(bbox_dir)
print(("bbox_dir = " + bbox_dir))

In [ ]:
mask_dir = os.path.join(os.getcwd(), '../data/target/mask')
mask_dir = os.path.abspath(mask_dir)
print(("mask_dir = " + mask_dir))

In [ ]:
ref_dir = os.path.join(os.getcwd(), '../data/reference/bbox')
ref_dir = os.path.abspath(ref_dir)
print(("ref_dir = " + ref_dir))

## Loads ICBM2009 referential

In [ ]:
install_dir = "."
extracted_dir = f"{install_dir}/mni_icbm152_nlin_asym_09c"
if os.path.exists(extracted_dir):
    print(f'the directory {extracted_dir} already exists. Assuming it is OK.')
else:
    dl_url = "http://www.bic.mni.mcgill.ca/~vfonov/icbm/2009/mni_icbm152_nlin_asym_09c_nifti.zip"
    tmp_dl = tempfile.mkstemp(suffix='.zip')
    with urlopen(dl_url) as f:
        with open(tmp_dl[1], 'wb') as g:
            g.write(f.read())
    # Extract the archive
    with zipfile.ZipFile(tmp_dl[1], 'r') as zf:
        zf.extractall(install_dir)

In [ ]:
mni_file = f"{extracted_dir}/mni_icbm152_t1_tal_nlin_asym_09c.nii"
vol_mri = aims.read(mni_file)
vol_mri.header()

# Creates a mask with one subject

In [ ]:
unsupervised_dir = os.path.join(os.getcwd(), '../data/source/unsupervised')
reference_dir = os.path.join(os.getcwd(), '../data/reference')
src_dir_subject = f"{unsupervised_dir}/ANALYSIS/3T_morphologist"
tgt_dir = os.path.join(os.getcwd(), '../data/target')
skeleton_raw_dir = f"{tgt_dir}/skeletons/raw"
transform_dir = f"{tgt_dir}/transforms"
skeleton_1mm_dir = f"{tgt_dir}/skeletons/1mm"

In [ ]:
from cortical_tiles.brainvisa import generate_skeletons
from cortical_tiles.brainvisa import generate_ICBM2009c_transforms
from cortical_tiles.brainvisa import resample_files
from cortical_tiles.brainvisa import compute_bounding_box
from cortical_tiles.brainvisa import compute_mask

In [ ]:
generate_skeletons.generate_skeletons(
    src_dir=src_dir_subject,
    skeleton_dir=skeleton_raw_dir,
    side=side)

In [ ]:
generate_ICBM2009c_transforms.generate_ICBM2009c_transforms(
    src_dir=src_dir_subject,
    transform_dir=transform_dir,
    side=side)

In [ ]:
resample_files.resample_files(
    src_dir=skeleton_raw_dir,
    input_type='skeleton',
    resampled_dir=skeleton_1mm_dir,
    transform_dir=transform_dir,
    side=side)

In [ ]:
compute_bounding_box.compute_bounding_box(
    src_dir=src_dir_supervised,
    bbox_dir=bbox_dir,
    sulcus=sulcus_post,
    side=side,
    out_voxel_size=1.)

Computes mask for sulcus sulcus_ant. The resulting volume will be used for plotting. The mask and the bounding box are also saved in files.

In [ ]:
vol_mask_one_subject = compute_mask.compute_mask(
    src_dir=src_dir,
    mask_dir=mask_dir,
    sulcus=sulcus_post,
    side=side,
    number_subjects=1,
    out_voxel_size=1)

In [ ]:
print(vol_mask_one_subject.shape) 
print(vol_mask_one_subject.header())

In [ ]:
# We test aims converters voluyme -> bucket -> volume

c = aims.Converter_rc_ptr_Volume_S16_BucketMap_VOID()

bck_mask_one_subject = c(vol_mask_one_subject)
c2 = aims.Converter(intype=bck_mask_one_subject, outtype='rc_ptr_Volume_S16')
vol_recovered = c2(bck_mask_one_subject)
print(np.unique((np.asarray(vol_mask_one_subject)), return_counts=True))
print(np.unique((np.asarray(vol_recovered)), return_counts=True))
mask_mesh_one_subject = dtx._aims_tools.bucket_to_mesh(bck_mask_one_subject[0])

# Defines plot functions

In [ ]:
# This is a global list to register fusion results
fusion2d = []

In [ ]:
# Defines the function to plot the volume in anatomist
def plot_axial(vol):
    global a
    print(vol.header())
    a_vol = a.toAObject(vol_recovered)
    axial = a.createWindow('Axial')
    axial.addObjects(a_vol)
    return axial

In [ ]:
# Defines the function to plot the mesh in anatomist
def plot_3D(mesh):
    global a
    a_mesh = a.toAObject(mesh)
    w3d = a.createWindow("3D")
    w3d.addObjects(a_mesh)
    return w3d

In [ ]:
# It makes the fusion of 2 volumes
def plot_fusion_2D(vol1, vol2):
    global a
    global fusion2d
    a_vol1 = a.toAObject(vol1)
    a_vol2 = a.toAObject(vol2)
    fusion2d.append(a.fusionObjects([a_vol1, a_vol2], "Fusion2DMethod"))
    axial = a.createWindow("Axial")
    axial.addObjects(fusion2d[-1])
    # params of the fusion : linear on non null
    a.execute("Fusion2DParams", object=fusion2d[-1], mode="linear_on_defined", rate=0.4)
    return axial

In [ ]:
def plot_3D_with_white_mesh(mesh, white_mesh): 
    global a
    material = a.Material(diffuse=[0.8, 0.8, 0.8, 0.8])
    material_strong = a.Material(diffuse=[1., 0.6, 0.6, 0.6])
    a_mesh = a.toAObject(mesh)
    a_white_mesh = a.toAObject(white_mesh)
    a_white_mesh.setMaterial(material)
    a_mesh.setMaterial(material=material_strong)
    w3d = a.createWindow("3D")
    w3d.addObjects([a_mesh, a_white_mesh])
    return w3d

In [ ]:
def plot_3D_box_white_mesh(mesh_box, mesh_white, mesh):
    global a
    # Defines material
    material_box = a.Material(diffuse=[0.8, 0.6, 0., 0.6])
    material_white = a.Material(diffuse=[0.8, 0.8, 0.8, 0.8])
    material_mesh = a.Material(diffuse=[1, 0.6, 0.6, 0.6])
    
    # Transforms aims objects into anatomist objects
    a_mesh_box = a.toAObject(mesh_box)
    a_mesh_white = a.toAObject(mesh_white)
    a_mesh = a.toAObject(mesh)
    
    # Sets material
    a_mesh_box.setMaterial(material=material_box)
    a_mesh_white.setMaterial(material=material_white)
    a_mesh.setMaterial(material=material_mesh)
    
    # Creates window
    w3d = a.createWindow("3D")
    w3d.addObjects([a_mesh_box, a_mesh_white, a_mesh])
    return w3d

# Plottings

In [ ]:
fig1 = plot_axial(vol_recovered)

Another bucket converter:

In [ ]:
# Another bucket converter
temp_dir = tempfile.mkdtemp()
mask_filename = f"{temp_dir}/mask.nii.gz"
aims.write(vol_mask_one_subject, mask_filename)
bucket_filename = f"{temp_dir}/mask.bck"
cmd = f"AimsFileConvert -c Bucket -t VOID -e 1 -i {mask_filename} -o {bucket_filename}"
os.system(cmd)

# Loads bucket file
bucket, bucket_raw, dxyz, rot, tr = pcpm.load_bucket(bucket_filename)

# Displays bucket file
mask_mesh = dtx._aims_tools.bucket_to_mesh(bucket)
cld.draw(mask_mesh)

In [ ]:
fig2 = plot_3D(mask_mesh)

In [ ]:
fig3 = plot_fusion_2D(vol_mri, vol_mask_one_subject)

# Plots mask mesh together with white mesh

We take the white matter mesh:

In [ ]:
src_dir = os.path.join(os.getcwd(), '../data/source/supervised')
src_dir = os.path.abspath(src_dir)
subject_dir = f"{src_dir}/sujet01/t1mri/t1"
src_mesh = f"{subject_dir}/default_analysis/segmentation/mesh"
mesh_white_file = f"{src_mesh}/sujet01_Lwhite.mesh"
print(os.path.isfile(mesh_white_file))

We transform it in MNI152 referential space:

In [ ]:
graph_file = f"{subject_dir}/default_analysis/folds/3.3/base2018_manual/Lsujet01_base2018_manual.arg"
graph = aims.read(graph_file)
g_to_icbm_template = aims.GraphManip.getICBM2009cTemplateTransform(graph)
file_g_to_icbm_template = f"{temp_dir}/file_g_to_icbm.trm"
aims.write(g_to_icbm_template, file_g_to_icbm_template)
mesh_white_file_mni2009 = f"{temp_dir}/sujet01_Lwhite_mni2009.mesh"

In [ ]:
cmd = 'AimsApplyTransform' + \
    ' -i ' + mesh_white_file + \
    ' -o ' + mesh_white_file_mni2009 + \
    ' -m ' + file_g_to_icbm_template + \
    ' -r ' + mni_file + \
    ' -t linear'
os.system(cmd)

In [ ]:
mesh_white = aims.read(mesh_white_file_mni2009)

We represent the mask mesh together with the white mesh of the reference MNI152 2009c:

In [ ]:
fig4 = plot_3D_with_white_mesh(mask_mesh, mesh_white)

# Crops based upon bounding box

In [ ]:
interp = 'nearest' # note: no effect if resampling is not none
resampling = 's' # sulcus-based
list_sulci = [sulcus_ant, sulcus_post]
crop_dir = os.path.join(os.getcwd(), '../data/target/crops/sulcus_based')
crop_dir = os.path.abspath(crop_dir)
print(("crop_dir = " + crop_dir))

In [ ]:
from cortical_tiles.brainvisa import generate_crops

generate_crops.generate_crops(
    src_dir=skeleton_1mm_dir,
    crop_dir=crop_dir,
    bbox_dir=bbox_dir,
    cropping_type='bbox',
    list_sulci=list_sulci,
    side=side,
    number_subjects=1)

In [ ]:
crop_file = glob.glob(crop_dir + '/Lcrops/*.nii.gz')[0]
print(crop_file)
vol_crop = aims.read(crop_file)
dict(vol_crop.header())

In [ ]:
fig8 = plot_axial(vol_crop)

We now shift the position using the bouding box information (here done by hand)

In [ ]:
tr = aims.AffineTransformation3d(aims.Quaternion([0, 0, 0, 1]))
tr.setTranslation((130, 140, 61))
print(tr)

vol_shifted = aims.Volume(vol_mri.header()['volume_dimension'], dtype=vol_crop.__array__().dtype)
vol_shifted.header()['voxel_size'] = vol_mri.header()['voxel_size']

resampler = aimsalgo.ResamplerFactory(vol_crop).getResampler(0)
resampler.setDefaultValue(11)
resampler.setRef(vol_crop)
resampler.resample(vol_crop, tr, 0, vol_shifted)
print(np.unique(np.array(vol_shifted)))

In [ ]:
# We test aims converters volume -> bucket -> mesh
vol_dir = f'{temp_dir}/vol'
crop_dir_shifted = vol_dir + '/Lcrops'
if os.path.exists(crop_dir_shifted):
    shutil.rmtree(crop_dir_shifted)
os.makedirs(crop_dir_shifted, exist_ok=True)
aims.write(vol_shifted, f'{crop_dir_shifted}/100206_normalized.nii.gz')
print(glob.glob(crop_dir_shifted + '/*'))
mesh_dir = f'{temp_dir}/mesh'
if os.path.exists(mesh_dir):
    shutil.rmtree(mesh_dir)
from cortical_tiles.brainvisa.utils import remove_hull

d = remove_hull.DatasetHullRemoved(src_dir=vol_dir, tgt_dir=mesh_dir, side='L', number_subjects=1)

bck_shifted = d.create_meshes()

In [ ]:
bck_shifted

In [ ]:
mesh_shifted = dtx._aims_tools.bucket_to_mesh(bck_shifted['100206'])

We now read the mesh white file of the subject

In [ ]:
subject_dir = f'{unsupervised_dir}/ANALYSIS/3T_morphologist/100206/t1mri/default_acquisition'
graph_file = f"{subject_dir}/default_analysis/folds/3.1/default_session_auto/L100206_default_session_auto.arg"
mesh_white_file_100206 = f"{subject_dir}/default_analysis/segmentation/mesh/100206_Lwhite.gii"
graph = aims.read(graph_file)
g_to_icbm_template = aims.GraphManip.getICBM2009cTemplateTransform(graph)
file_g_to_icbm_template = f"{temp_dir}/file_g_to_icbm.trm"
aims.write(g_to_icbm_template, file_g_to_icbm_template)
mesh_white_file_100206_mni2009 = f"{temp_dir}/100206_Lwhite_mni2009.mesh"

In [ ]:
cmd = 'AimsApplyTransform' + \
    ' -i ' + mesh_white_file_100206 + \
    ' -o ' + mesh_white_file_100206_mni2009 + \
    ' -m ' + file_g_to_icbm_template + \
    ' -r ' + mni_file + \
    ' -t linear'
os.system(cmd)

In [ ]:
mesh_white_100206 = aims.read(mesh_white_file_100206_mni2009)

In [ ]:
fig9 = plot_3D_with_white_mesh(mesh=mesh_shifted, white_mesh=mesh_white_100206)

In [ ]:
fig = cld.draw(mesh_shifted, color='blue')
cld.draw(mesh_white_100206, color='red', fig=fig)
#cld.draw([mesh_shifted, mesh_white_100206], color=['blue', 'red'])

## Visualize crop box in transparency together with mask mesh and white mesh

In [ ]:
import json
bbox_json_file = "../data/target/bbox/L/S.T.s.ter.asc.ant._left.json"
with open(bbox_json_file) as json_file:
    bbox = json.load(json_file)
bbox

In [ ]:
size_box = (np.array(bbox['bbmax_voxel']) - np.array(bbox['bbmin_voxel'])).astype(int)
size_box = np.concatenate([size_box, [1]])
size_box

In [ ]:
mesh_box = aims.SurfaceGenerator.parallelepiped(bbox['bbmin_voxel'], bbox['bbmax_voxel'])

In [ ]:
cld.draw(mesh_box)

In [ ]:
fig7 = plot_3D_box_white_mesh(mesh_box=mesh_box,
                              mesh_white=mesh_white,
                              mesh=mask_mesh)